# IBM AML Demo Queries

This notebook runs Gremlin queries and shows query results.
Start backend first in another terminal:
`mvn exec:java`


In [1]:
import requests
import pandas as pd
import subprocess
from typing import Dict, Any
from IPython.display import display, Markdown
from pathlib import Path

BASE_URL = "http://localhost:7000"
MAX_ROWS = 100000
REPO_ROOT = Path.home() / "SourceCode/graph-query-engine"


def run_aml_data_download(variant: str = "HI-Small", rows: int = 100000) -> bool:
    # Download HI-Small files and generate demo/data/aml-demo.csv via normalize_aml.py.
    if not REPO_ROOT.exists():
        display(Markdown(f"**Error:** repo path not found: `{REPO_ROOT}`"))
        return False

    cmd = f"bash ./scripts/download_aml_data.sh --variant {variant} --rows {rows}"
    display(Markdown(f"Running: `{cmd}` in `{REPO_ROOT}`"))
    proc = subprocess.run(cmd, cwd=REPO_ROOT, shell=True, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    return proc.returncode == 0


def resolve_csv_path() -> str | None:
    candidates = [
        Path.cwd() / "demo/data/aml-demo.csv",
        Path.cwd() / "demo/data/HI-Small_Trans.csv",
        Path.cwd() / "data/aml-normalized.csv",
        Path.cwd() / "data/aml-demo.csv",
        Path.cwd() / "aml-demo.csv",
    ]
    for p in candidates:
        if p.exists():
            return str(p)

    # Fallback to any normalized or raw transaction CSV in demo/data.
    demo_data = Path.cwd() / "demo/data"
    if demo_data.exists():
        for pat in ["*aml*.csv", "*Trans.csv"]:
            matches = sorted(demo_data.glob(pat))
            if matches:
                return str(matches[0])
    return None


CSV_PATH = resolve_csv_path()
print("BASE_URL:", BASE_URL)
print("CSV_PATH:", CSV_PATH)
print("MAX_ROWS:", MAX_ROWS)


BASE_URL: http://localhost:7000
CSV_PATH: /Users/vjoshi/SourceCode/graph-query-engine/demo/data/aml-demo.csv
MAX_ROWS: 100000


## Step 1: Prepare CSV (HI-Small)

If `demo/data/aml-demo.csv` is missing, run this in terminal:

```zsh
cd ~/SourceCode/graph-query-engine
bash ./scripts/download_aml_data.sh --variant HI-Small --rows 100000
```

This downloads `HI-Small_Trans.csv` and creates normalized `demo/data/aml-demo.csv`.

You can also run the helper below from notebook.


In [2]:
AUTO_RUN_DOWNLOAD = False
DOWNLOAD_VARIANT = "HI-Small"
DOWNLOAD_ROWS = 100000

if AUTO_RUN_DOWNLOAD:
    ok = run_aml_data_download(variant=DOWNLOAD_VARIANT, rows=DOWNLOAD_ROWS)
    if ok:
        CSV_PATH = resolve_csv_path()
        print("CSV_PATH refreshed:", CSV_PATH)
else:
    print("Set AUTO_RUN_DOWNLOAD=True to run download + normalize from notebook.")


Set AUTO_RUN_DOWNLOAD=True to run download + normalize from notebook.


In [3]:
def get_sql_explain(gremlin_query: str, include_plan: bool = False) -> Dict[str, Any]:
    try:
        response = requests.post(
            f"{BASE_URL}/query/explain",
            json={"gremlin": gremlin_query},
            headers={"Content-Type": "application/json"},
            params={"plan": "true"} if include_plan else {},
            timeout=10,
        )
        if response.ok:
            return response.json()
        return {"error": f"HTTP {response.status_code}: {response.text}"}
    except Exception as e:
        return {"error": str(e)}


def run_gremlin_query(gremlin_query: str, tx_mode: bool = False, show_plan: bool = False) -> Dict[str, Any]:
    endpoint = "/gremlin/query/tx" if tx_mode else "/gremlin/query"
    result: Dict[str, Any] = {}
    try:
        response = requests.post(
            f"{BASE_URL}{endpoint}",
            json={"gremlin": gremlin_query},
            headers={"Content-Type": "application/json"},
            timeout=60,
        )
        result = response.json()
    except Exception as e:
        result = {"error": str(e)}
    result["_sql_explain"] = get_sql_explain(gremlin_query, include_plan=show_plan)
    return result


def _render_plan(plan: dict) -> str:
    """Format a QueryPlan dict as compact readable markdown."""
    Q = "`"
    lines = []
    lines.append(
        f"- **rootType:** {Q}{plan.get('rootType')}{Q}"
        f"  **rootLabel:** {Q}{plan.get('rootLabel')}{Q}"
        f"  **rootTable:** {Q}{plan.get('rootTable')}{Q}"
    )
    lines.append(f"- **dialect:** {Q}{plan.get('dialect')}{Q}")
    if plan.get("rootIdFilter"):
        lines.append(f"- **rootIdFilter:** {Q}{plan['rootIdFilter']}{Q}")
    if plan.get("filters"):
        f_parts = [f"{Q}{f['property']} {f['operator']} {f['value']}{Q}" for f in plan["filters"]]
        lines.append("- **filters:** " + ", ".join(f_parts))
    for h in plan.get("hops") or []:
        labels = ", ".join(h.get("labels") or [])
        et = h.get("resolvedEdgeTable", "?")
        tl = h.get("resolvedTargetLabel", "?")
        tt = h.get("resolvedTargetTable", "?")
        lines.append(f"- **hop:** {Q}{h['direction']}({labels}){Q} \u2192 edge={Q}{et}{Q} \u2192 target={Q}{tl}{Q} ({Q}{tt}{Q})")
    if plan.get("aggregation"):
        prop = plan.get("aggregationProperty")
        lines.append(f"- **aggregation:** {Q}{plan['aggregation']}{Q}" + (f" on {Q}{prop}{Q}" if prop else ""))
    if plan.get("valuesProperty"):
        lines.append(f"- **valuesProperty:** {Q}{plan['valuesProperty']}{Q}")
    if plan.get("projections"):
        p_parts = [f"{Q}{p['alias']}{Q} ({p['kind']})" for p in plan["projections"]]
        lines.append("- **projections:** " + ", ".join(p_parts))
    if plan.get("orderBy"):
        lines.append(f"- **orderBy:** {Q}{plan['orderBy']}{Q} {Q}{plan.get('orderDirection', 'ASC')}{Q}")
    if plan.get("limit") is not None:
        lines.append(f"- **limit:** {Q}{plan['limit']}{Q}")
    if plan.get("preHopLimit") is not None:
        lines.append(f"- **preHopLimit:** {Q}{plan['preHopLimit']}{Q}")
    if plan.get("simplePath"):
        lines.append(f"- **simplePath:** {Q}true{Q}")
    if plan.get("dedup"):
        lines.append(f"- **dedup:** {Q}true{Q}")
    if plan.get("where"):
        w = plan["where"]
        lines.append(f"- **where:** {Q}{w['kind']}{Q} left={Q}{w.get('left')}{Q} right={Q}{w.get('right')}{Q}")
    if plan.get("asAliases"):
        a_parts = [f"{Q}{a['label']}{Q}@hop{a['hopIndexAfter']}" for a in plan["asAliases"]]
        lines.append("- **aliases:** " + ", ".join(a_parts))
    return "\n".join(lines)


def display_query_result(gremlin: str, result: Dict[str, Any], title: str = "", limit: int = 10, tx_mode: bool = False):
    if title:
        display(Markdown(f"### {title}"))
    display(Markdown("**Gremlin:**"))
    display(Markdown(f"```groovy\n{gremlin}\n```"))

    explain = result.get("_sql_explain", {})
    if "error" in explain:
        display(Markdown(f"**SQL Translation:** *not available ({explain['error']})*"))
    else:
        sql = explain.get("translatedSql", "")
        params = explain.get("parameters", [])
        plan = explain.get("plan")
        if sql:
            display(Markdown("**SQL Translation:**"))
            display(Markdown(f"```sql\n{sql}\n```"))
            if params:
                display(Markdown(f"**Parameters:** `{params}`"))
        if plan:
            display(Markdown("**Query Plan:**"))
            display(Markdown(_render_plan(plan)))

    if "error" in result:
        display(Markdown(f"**Error:** {result['error']}"))
        return

    rows = result.get("results", [])
    display(Markdown(f"**Result Count:** {result.get('resultCount', 0)}"))
    if not rows:
        return
    if isinstance(rows[0], dict):
        display(pd.DataFrame(rows[:limit]))
    else:
        for i, row in enumerate(rows[:limit], 1):
            print(f"{i}. {row}")


## Step 0: Health check


In [4]:
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5).text
    provider = requests.get(f"{BASE_URL}/gremlin/provider", timeout=5).json().get("provider", "unknown")
    display(Markdown(f"Status: `{health}`"))
    display(Markdown(f"Provider: `{provider}`"))
except Exception as e:
    display(Markdown(f"Health check failed: {e}"))


Status: `{"service":"graph-query-engine","status":"ok"}`

Provider: `tinkergraph`

## Step 2: Load AML CSV and Mapping


In [5]:
if not CSV_PATH:
    display(Markdown("**CSV not found.**"))
    display(Markdown('''Expected one of:
- `demo/data/aml-demo.csv`
- `demo/data/HI-Small_Trans.csv`
- `data/aml-normalized.csv`
- `data/aml-demo.csv`'''))
    display(Markdown('''Use:
```zsh
cd ~/SourceCode/graph-query-engine
bash ./scripts/download_aml_data.sh --variant HI-Small --rows 100000
```'''))
else:
    response = requests.post(
        f"{BASE_URL}/admin/load-aml-csv",
        params={"path": CSV_PATH, "maxRows": MAX_ROWS},
        timeout=120,
    )
    print(response.json())


{'rowsLoaded': 100000, 'accountsCreated': 81352, 'banksCreated': 4598, 'countriesCreated': 10, 'transactionsCreated': 100000, 'alertsCreated': 5, 'transfersCreated': 100000, 'sourcePath': '/Users/vjoshi/SourceCode/graph-query-engine/demo/data/aml-demo.csv', 'maxRows': 100000, 'provider': 'tinkergraph'}


## Query Sections


## Simple Queries


### S1 Count accounts

Count unique Account vertices.


In [6]:
gremlin = "g.V().hasLabel('Account').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S1 Count accounts', limit=10, tx_mode=False)


### S1 Count accounts

**Gremlin:**

```groovy
g.V().hasLabel('Account').count()
```

**SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml_accounts
```

**Result Count:** 1

1. 81352


### S2 Count banks

Count Bank vertices - one per distinct bank ID in the data.


In [7]:
gremlin = "g.V().hasLabel('Bank').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S2 Count banks', limit=10, tx_mode=False)


### S2 Count banks

**Gremlin:**

```groovy
g.V().hasLabel('Bank').count()
```

**SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml_banks
```

**Result Count:** 1

1. 4598


### S3 Count countries

Count Country vertices (10 pre-seeded jurisdictions).


In [8]:
gremlin = "g.V().hasLabel('Country').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S3 Count countries', limit=10, tx_mode=False)


### S3 Count countries

**Gremlin:**

```groovy
g.V().hasLabel('Country').count()
```

**SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml_countries
```

**Result Count:** 1

1. 10


### S4 Count alerts

Count Alert vertices - one raised per suspicious transfer.


In [9]:
gremlin = "g.V().hasLabel('Alert').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S4 Count alerts', limit=10, tx_mode=False)


### S4 Count alerts

**Gremlin:**

```groovy
g.V().hasLabel('Alert').count()
```

**SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml_alerts
```

**Result Count:** 1

1. 5


### S5 Count transfers

Count all TRANSFER edges.


In [10]:
gremlin = "g.E().hasLabel('TRANSFER').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S5 Count transfers', limit=10, tx_mode=False)


### S5 Count transfers

**Gremlin:**

```groovy
g.E().hasLabel('TRANSFER').count()
```

**SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml_transfers
```

**Result Count:** 1

1. 100000


### S6 Suspicious transfer count

Count confirmed suspicious TRANSFER edges (isLaundering=1).


In [11]:
gremlin = "g.E().has('isLaundering','1').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S6 Suspicious transfer count', limit=10, tx_mode=False)


### S6 Suspicious transfer count

**Gremlin:**

```groovy
g.E().has('isLaundering','1').count()
```

**SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml_transfers WHERE is_laundering = ?
```

**Parameters:** `['1']`

**Result Count:** 1

1. 5


### S7 High-risk countries

List Country vertices with riskLevel=HIGH.


In [12]:
gremlin = "g.V().hasLabel('Country').has('riskLevel','HIGH').project('countryCode','countryName','region','fatfBlacklist').by('countryCode').by('countryName').by('region').by('fatfBlacklist')"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S7 High-risk countries', limit=10, tx_mode=False)


### S7 High-risk countries

**Gremlin:**

```groovy
g.V().hasLabel('Country').has('riskLevel','HIGH').project('countryCode','countryName','region','fatfBlacklist').by('countryCode').by('countryName').by('region').by('fatfBlacklist')
```

**SQL Translation:**

```sql
SELECT v.country_code AS "countryCode", v.country_name AS "countryName", v.region AS "region", v.fatf_blacklist AS "fatfBlacklist" FROM aml_countries v WHERE v.risk_level = ?
```

**Parameters:** `['HIGH']`

**Result Count:** 3

,countryCode,countryName,region,fatfBlacklist
0,NG,Nigeria,Africa,false
1,KY,Cayman Islands,Americas,true
2,PA,Panama,Americas,true


### S8 High-severity alerts

Show HIGH-severity open alerts.


In [13]:
gremlin = "g.V().hasLabel('Alert').has('severity','HIGH').project('alertId','alertType','status','raisedAt').by('alertId').by('alertType').by('status').by('raisedAt').limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S8 High-severity alerts', limit=15, tx_mode=False)


### S8 High-severity alerts

**Gremlin:**

```groovy
g.V().hasLabel('Alert').has('severity','HIGH').project('alertId','alertType','status','raisedAt').by('alertId').by('alertType').by('status').by('raisedAt').limit(15)
```

**SQL Translation:**

```sql
SELECT v.alert_id AS "alertId", v.alert_type AS "alertType", v.status AS "status", v.raised_at AS "raisedAt" FROM aml_alerts v WHERE v.severity = ? LIMIT 15
```

**Parameters:** `['HIGH']`

**Result Count:** 2

,alertId,alertType,status,raisedAt
0,ALERT-4743,SUSPICIOUS_TRANSFER,OPEN,2022/09/01 00:21
1,ALERT-85764,SUSPICIOUS_TRANSFER,OPEN,2022/09/01 00:03


## Complex Queries


### C1 Top sender accounts

Accounts ranked by outgoing transfer count - find the biggest hubs.


In [14]:
gremlin = "g.V().hasLabel('Account').project('accountId','bankId','outDegree').by('accountId').by('bankId').by(outE('TRANSFER').count()).order().by(select('outDegree'),Order.desc).limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C1 Top sender accounts', limit=15, tx_mode=False)


### C1 Top sender accounts

**Gremlin:**

```groovy
g.V().hasLabel('Account').project('accountId','bankId','outDegree').by('accountId').by('bankId').by(outE('TRANSFER').count()).order().by(select('outDegree'),Order.desc).limit(15)
```

**SQL Translation:**

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT COUNT(*) FROM aml_transfers WHERE from_account_id = v.id) AS "outDegree" FROM aml_accounts v ORDER BY "outDegree" DESC LIMIT 15
```

**Result Count:** 15

,accountId,bankId,outDegree
0,100428660,070,1924
1,8001409E0,001,13
2,80026D340,010,13
3,800631890,01601,13
4,8001C3570,010,12
5,8008F0D20,021174,12
6,8001523C0,010,11
7,8001A4930,012,11
8,8001F3760,010,11
9,800537DE0,001,11


### C2 Suspicious hubs

Accounts with the most suspicious outgoing transfers.


In [15]:
gremlin = "g.V().hasLabel('Account').project('accountId','bankId','suspiciousOut','totalOut').by('accountId').by('bankId').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).where(select('suspiciousOut').is(gt(0))).order().by(select('suspiciousOut'),Order.desc).limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C2 Suspicious hubs', limit=15, tx_mode=False)


### C2 Suspicious hubs

**Gremlin:**

```groovy
g.V().hasLabel('Account').project('accountId','bankId','suspiciousOut','totalOut').by('accountId').by('bankId').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).where(select('suspiciousOut').is(gt(0))).order().by(select('suspiciousOut'),Order.desc).limit(15)
```

**SQL Translation:**

```sql
SELECT * FROM (SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT COUNT(*) FROM aml_transfers WHERE from_account_id = v.id AND is_laundering = '1') AS "suspiciousOut", (SELECT COUNT(*) FROM aml_transfers WHERE from_account_id = v.id) AS "totalOut" FROM aml_accounts v) p WHERE p."suspiciousOut" > 0 ORDER BY "suspiciousOut" DESC LIMIT 15
```

**Result Count:** 1

,accountId,bankId,suspiciousOut,totalOut
0,100428660,070,5,1924


### C3 Account -> Bank (BELONGS_TO)

Show which bank each account belongs to via BELONGS_TO.


In [16]:
gremlin = "g.V().hasLabel('Account').limit(15).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C3 Account -> Bank (BELONGS_TO)', limit=15, tx_mode=False)


### C3 Account -> Bank (BELONGS_TO)

**Gremlin:**

```groovy
g.V().hasLabel('Account').limit(15).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())
```

**SQL Translation:**

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT STRING_AGG(_njv0.bank_name, ',') FROM aml_account_bank _nje0 JOIN aml_banks _njv0 ON _njv0.id = _nje0.bank_id WHERE _nje0.account_id = v.id) AS "bankName" FROM aml_accounts v LIMIT 15
```

**Result Count:** 15

,accountId,bankId,bankName
0,8000EBD30,010,[Bank-010]
1,8000F4580,03208,[Bank-03208]
2,8000F5340,001,[Bank-001]
3,8000F4670,03209,[Bank-03209]
4,8000F5030,012,[Bank-012]
5,8000F5200,010,[Bank-010]
6,8000F5AD0,001,[Bank-001]
7,8000EBAC0,001,[Bank-001]
8,8000EC1E0,001,[Bank-001]
9,8000EC280,012,[Bank-012]


### C4 Bank -> Country (LOCATED_IN)

Show which country each bank is headquartered in.


In [17]:
gremlin = "g.V().hasLabel('Bank').limit(15).project('bankId','bankName','countryCode','countryName').by('bankId').by('bankName').by('countryCode').by(out('LOCATED_IN').values('countryName').fold())"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C4 Bank -> Country (LOCATED_IN)', limit=15, tx_mode=False)


### C4 Bank -> Country (LOCATED_IN)

**Gremlin:**

```groovy
g.V().hasLabel('Bank').limit(15).project('bankId','bankName','countryCode','countryName').by('bankId').by('bankName').by('countryCode').by(out('LOCATED_IN').values('countryName').fold())
```

**SQL Translation:**

```sql
SELECT v.bank_id AS "bankId", v.bank_name AS "bankName", v.country_code AS "countryCode", (SELECT STRING_AGG(_njv0.country_name, ',') FROM aml_bank_country _nje0 JOIN aml_countries _njv0 ON _njv0.id = _nje0.country_id WHERE _nje0.bank_id = v.id) AS "countryName" FROM aml_banks v LIMIT 15
```

**Result Count:** 15

,bankId,bankName,countryCode,countryName
0,010,Bank-010,SG,[Singapore]
1,03208,Bank-03208,CH,[Switzerland]
2,001,Bank-001,SG,[Singapore]
3,03209,Bank-03209,HK,[Hong Kong]
4,012,Bank-012,NG,[Nigeria]
5,002439,Bank-002439,AE,[UAE]
6,0211050,Bank-0211050,SG,[Singapore]
7,011813,Bank-011813,DE,[Germany]
8,0245335,Bank-0245335,KY,[Cayman Islands]
9,0036056,Bank-0036056,AE,[UAE]


### C5 Two-hop: Account -> Bank -> Country

Traverse Account->Bank->Country in two hops.


In [18]:
gremlin = "g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C5 Two-hop: Account -> Bank -> Country', limit=10, tx_mode=False)


### C5 Two-hop: Account -> Bank -> Country

**Gremlin:**

```groovy
g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)
```

**SQL Translation:**

```sql
SELECT v0.account_id AS accountId0, v1.bank_name AS bankName1, v2.country_name AS countryName2 FROM (SELECT * FROM aml_accounts LIMIT 1) v0 JOIN aml_account_bank e1 ON e1.account_id = v0.id JOIN aml_banks v1 ON v1.id = e1.bank_id JOIN aml_bank_country e2 ON e2.bank_id = v1.id JOIN aml_countries v2 ON v2.id = e2.country_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) LIMIT 10
```

**Result Count:** 1

1. ['8000EBD30', 'Bank-010', 'Singapore']


### C6 Accounts sending to high-risk countries (SENT_VIA)

Find accounts routing money via FATF-blacklisted countries.


In [19]:
gremlin = "g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(20)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C6 Accounts sending to high-risk countries (SENT_VIA)', limit=20, tx_mode=False)


### C6 Accounts sending to high-risk countries (SENT_VIA)

**Gremlin:**

```groovy
g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(20)
```

**SQL Translation:**

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", v.risk_score AS "riskScore" FROM aml_accounts v WHERE EXISTS (SELECT 1 FROM aml_transfer_channel _we JOIN aml_countries _wv ON _wv.id = _we.country_id WHERE _we.transfer_id = v.id AND _wv.fatf_blacklist = ?) LIMIT 20
```

**Parameters:** `['true']`

**Result Count:** 20

,accountId,bankId,riskScore
0,8000F4FE0,001,0.15
1,8005E18F0,01665,0.15
2,8005E24C0,01665,0.15
3,800058100,010,0.15
4,800AC8E30,01674,0.15
5,800AC9010,021611,0.15
6,80005D5C0,010,0.15
7,801009860,01674,0.15
8,80100A2C0,001411,0.10
9,8012275D0,001,0.15


### C7 Flagged accounts with alert detail (FLAGGED_BY)

Show investigated accounts with linked Alert severity.


In [20]:
gremlin = "g.V().hasLabel('Account').where(outE('FLAGGED_BY')).project('accountId','bankId','alertCount','highSeverity').by('accountId').by('bankId').by(outE('FLAGGED_BY').count()).by(out('FLAGGED_BY').has('severity','HIGH').count()).limit(20)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C7 Flagged accounts with alert detail (FLAGGED_BY)', limit=20, tx_mode=False)


### C7 Flagged accounts with alert detail (FLAGGED_BY)

**Gremlin:**

```groovy
g.V().hasLabel('Account').where(outE('FLAGGED_BY')).project('accountId','bankId','alertCount','highSeverity').by('accountId').by('bankId').by(outE('FLAGGED_BY').count()).by(out('FLAGGED_BY').has('severity','HIGH').count()).limit(20)
```

**SQL Translation:**

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT COUNT(*) FROM aml_account_alert WHERE account_id = v.id) AS "alertCount", (SELECT COUNT(*) FROM aml_account_alert _e JOIN aml_alerts _tv ON _tv.id = _e.alert_id WHERE _e.account_id = v.id AND _tv.severity = 'HIGH') AS "highSeverity" FROM aml_accounts v WHERE EXISTS (SELECT 1 FROM aml_account_alert we WHERE we.account_id = v.id) LIMIT 20
```

**Result Count:** 1

,accountId,bankId,alertCount,highSeverity
0,100428660,070,5,2


### C8 Cross-bank suspicious flow

Suspicious transfers that cross bank boundaries.


In [21]:
gremlin = "g.E().has('isLaundering','1').project('fromBank','fromAcct','toBank','toAcct','amount','currency').by(outV().values('bankId')).by(outV().values('accountId')).by(inV().values('bankId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C8 Cross-bank suspicious flow', limit=15, tx_mode=False)


### C8 Cross-bank suspicious flow

**Gremlin:**

```groovy
g.E().has('isLaundering','1').project('fromBank','fromAcct','toBank','toAcct','amount','currency').by(outV().values('bankId')).by(outV().values('accountId')).by(inV().values('bankId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)
```

**SQL Translation:**

```sql
SELECT ov.bank_id AS "fromBank", ov.account_id AS "fromAcct", iv.bank_id AS "toBank", iv.account_id AS "toAcct", e.amount AS "amount", e.currency AS "currency" FROM aml_transfers e JOIN aml_accounts ov ON ov.id = e.from_account_id JOIN aml_accounts iv ON iv.id = e.to_account_id WHERE e.is_laundering = ? LIMIT 15
```

**Parameters:** `['1']`

**Result Count:** 5

,fromBank,fromAcct,toBank,toAcct,amount,currency
0,070,100428660,001124,800825340,389769.39,US Dollar
1,070,100428660,011474,805B716C0,29024.33,US Dollar
2,070,100428660,015980,80B39E7B0,792.92,US Dollar
3,070,100428660,0113798,80DC756E0,13171425.53,US Dollar
4,070,100428660,0032375,80E480620,14288.83,US Dollar


### C9 Three-hop money trail

Follow suspicious TRANSFER hops 3 steps deep.


In [22]:
gremlin = "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C9 Three-hop money trail', limit=10, tx_mode=False)


### C9 Three-hop money trail

**Gremlin:**

```groovy
g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)
```

**SQL Translation:**

```sql
SELECT v0.account_id AS accountId0, v1.account_id AS accountId1, v2.account_id AS accountId2, v3.account_id AS accountId3 FROM (SELECT * FROM aml_accounts WHERE EXISTS (SELECT 1 FROM aml_transfers we WHERE we.from_account_id = id AND we.is_laundering = ?) LIMIT 1) v0 JOIN aml_transfers e1 ON e1.from_account_id = v0.id JOIN aml_accounts v1 ON v1.id = e1.to_account_id JOIN aml_transfers e2 ON e2.from_account_id = v1.id JOIN aml_accounts v2 ON v2.id = e2.to_account_id JOIN aml_transfers e3 ON e3.from_account_id = v2.id JOIN aml_accounts v3 ON v3.id = e3.to_account_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) AND v3.id NOT IN (v0.id, v1.id, v2.id) LIMIT 10
```

**Parameters:** `['1']`

**Result Count:** 10

1. ['100428660', '8068AC520', '8068AC570', '806DED3B0']
2. ['100428660', '8068AC520', '8068AC570', '806DED3B0']
3. ['100428660', '80025B860', '800423C60', '80192B4F0']
4. ['100428660', '80025B860', '800423C60', '80192B4F0']
5. ['100428660', '80025B860', '800423C60', '80192B4F0']
6. ['100428660', '80025B860', '800423C60', '80192B4F0']
7. ['100428660', '8005CD2C0', '8006DDC70', '800CA4130']
8. ['100428660', '8005CD2C0', '8006DDC70', '800CA4130']
9. ['100428660', '8005CD2C0', '8006DDC70', '800CA4130']
10. ['100428660', '8005CD2C0', '8006DDC70', '800CA4130']


### C10 Five-hop layering chain

Extended 5-hop traversal for layering detection.


In [23]:
gremlin = "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(5).path().by('accountId').limit(10)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C10 Five-hop layering chain', limit=10, tx_mode=False)


### C10 Five-hop layering chain

**Gremlin:**

```groovy
g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(5).path().by('accountId').limit(10)
```

**SQL Translation:**

```sql
SELECT v0.account_id AS accountId0, v1.account_id AS accountId1, v2.account_id AS accountId2, v3.account_id AS accountId3, v4.account_id AS accountId4, v5.account_id AS accountId5 FROM (SELECT * FROM aml_accounts WHERE EXISTS (SELECT 1 FROM aml_transfers we WHERE we.from_account_id = id AND we.is_laundering = ?) LIMIT 1) v0 JOIN aml_transfers e1 ON e1.from_account_id = v0.id JOIN aml_accounts v1 ON v1.id = e1.to_account_id JOIN aml_transfers e2 ON e2.from_account_id = v1.id JOIN aml_accounts v2 ON v2.id = e2.to_account_id JOIN aml_transfers e3 ON e3.from_account_id = v2.id JOIN aml_accounts v3 ON v3.id = e3.to_account_id JOIN aml_transfers e4 ON e4.from_account_id = v3.id JOIN aml_accounts v4 ON v4.id = e4.to_account_id JOIN aml_transfers e5 ON e5.from_account_id = v4.id JOIN aml_accounts v5 ON v5.id = e5.to_account_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) AND v3.id NOT IN (v0.id, v1.id, v2.id) AND v4.id NOT IN (v0.id, v1.id, v2.id, v3.id) AND v5.id NOT IN (v0.id, v1.id, v2.id, v3.id, v4.id) LIMIT 10
```

**Parameters:** `['1']`

**Result Count:** 0

### C11 Transactional suspicious count

Suspicious transfer count via transactional endpoint.


In [24]:
gremlin = "g.E().has('isLaundering','1').count()"
result = run_gremlin_query(gremlin, tx_mode=True)
display_query_result(gremlin, result, title='C11 Transactional suspicious count', limit=10, tx_mode=True)


### C11 Transactional suspicious count

**Gremlin:**

```groovy
g.E().has('isLaundering','1').count()
```

**SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml_transfers WHERE is_laundering = ?
```

**Parameters:** `['1']`

**Result Count:** 1

1. 5


## Iceberg Note

Use `aml_iceberg_queries.ipynb` for Iceberg comparisons.


## Done
